In [1]:
import pandas as pd 
import numpy as np 

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [3]:
import joblib
from pathlib import Path


In [4]:
df = pd.read_csv(r'../datasets/kidney_disease.csv')
print('Dataset loaded sucessfully')

Dataset loaded sucessfully


# dataset inspection 

In [5]:
print('Shape:', df.shape)

print('\n Columns:')
print(df.columns)

print('\n Dataset Information:')
df.info()

print('\n Missing values:')
df.isnull().sum()

print('\n First 5 rows:')
df.head()

Shape: (400, 26)

 Columns:
Index(['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr',
       'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')

 Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             391 non-null    float64
 2   bp              388 non-null    float64
 3   sg              353 non-null    float64
 4   al              354 non-null    float64
 5   su              351 non-null    float64
 6   rbc             248 non-null    object 
 7   pc              335 non-null    object 
 8   pcc             396 non-null    object 
 9   ba              396 non-null    object 
 10  bgr             356 non-null    float64
 11  bu              381 non-nul

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [6]:

# Count missing values in every column
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values[missing_values > 0].sort_values(ascending=False))

print("\nTotal missing values:", df.isnull().sum().sum())

print("\nTarget distribution:")
print(df["classification"].value_counts())

print("\nUnique target labels:")
print(df["classification"].unique())



# Inspect columns stored as objects
for col in ["pcv", "wc", "rc", "classification"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).head(15))


Missing values in each column:
rbc      152
rc       130
wc       105
pot       88
sod       87
pcv       70
pc        65
hemo      52
su        49
sg        47
al        46
bgr       44
bu        19
sc        17
bp        12
age        9
ba         4
pcc        4
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
dtype: int64

Total missing values: 1009

Target distribution:
classification
ckd       248
notckd    150
ckd\t       2
Name: count, dtype: int64

Unique target labels:
['ckd' 'ckd\t' 'notckd']

pcv:
pcv
NaN    70
52     21
41     21
48     19
44     19
40     16
43     14
42     13
45     13
36     12
33     12
28     12
50     12
32     12
37     11
Name: count, dtype: int64

wc:
wc
NaN      105
9800      11
6700      10
9600       9
7200       9
9200       9
6900       8
5800       8
11000      8
7800       7
7000       7
9400       7
9100       7
10500      6
6300       6
Name: count, dtype: int64

rc:
rc
NaN    130
5.2     18
4.5     16
4.9    

In [7]:
df = df.copy()

# Remove extra spaces and tab characters from column names
df.columns = df.columns.str.strip()

# Clean the target labels
df["classification"] = (
    df["classification"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("Target distribution after cleaning:")
print(df["classification"].value_counts())

# Convert these measurement columns to numerical types
numeric_columns = ["pcv", "wc", "rc"]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nCorrected data types:")
print(df[["pcv", "wc", "rc"]].dtypes)

print("\nMissing values after conversion:")
print(df[["pcv", "wc", "rc"]].isnull().sum())



# Separate input features and target
X = df.drop(columns=["id", "classification"])
y = df["classification"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature data types:")
print(X.dtypes.value_counts())

print("\nTarget labels:")
print(y.value_counts())


Target distribution after cleaning:
classification
ckd       250
notckd    150
Name: count, dtype: Int64

Corrected data types:
pcv    float64
wc     float64
rc     float64
dtype: object

Missing values after conversion:
pcv     71
wc     106
rc     131
dtype: int64
Feature shape: (400, 24)
Target shape: (400,)

Feature data types:
float64    14
object     10
Name: count, dtype: int64

Target labels:
classification
ckd       250
notckd    150
Name: count, dtype: Int64


In [8]:
# Identify numerical and categorical columns
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("\nNumber of numerical features:", len(numeric_features))

print("\nCategorical features:", categorical_features)
print("\nNumber of categorical features:", len(categorical_features))


Numerical features: ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']

Number of numerical features: 14

Categorical features: ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']

Number of categorical features: 10


In [9]:
# Convert target labels to binary values
y = y.map({"ckd": 1, "notckd": 0})

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


X_train: (320, 24)
X_test: (80, 24)
y_train: (320,)
y_test: (80,)

Training target distribution:
classification
1    200
0    120
Name: count, dtype: int64

Testing target distribution:
classification
1    50
0    30
Name: count, dtype: int64


In [10]:
# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully.")


Preprocessing pipeline created successfully.


In [11]:
# Logistic Regression pipeline
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

# Random Forest pipeline
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train both models
logistic_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

print("Both models trained successfully!")


Both models trained successfully!


In [12]:
# Generate predictions
lr_predictions = logistic_model.predict(X_test)
rf_predictions = rf_model.predict(X_test)

# Logistic Regression evaluation
print("=" * 45)
print("LOGISTIC REGRESSION")
print("=" * 45)

print("Accuracy:", round(
    accuracy_score(y_test, lr_predictions) * 100, 2
), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    lr_predictions,
    target_names=["Not CKD", "CKD"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, lr_predictions))


# Random Forest evaluation
print("\n" + "=" * 45)
print("RANDOM FOREST")
print("=" * 45)

print("Accuracy:", round(
    accuracy_score(y_test, rf_predictions) * 100, 2
), "%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    rf_predictions,
    target_names=["Not CKD", "CKD"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_predictions))


LOGISTIC REGRESSION
Accuracy: 98.75 %

Classification Report:
              precision    recall  f1-score   support

     Not CKD       0.97      1.00      0.98        30
         CKD       1.00      0.98      0.99        50

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80

Confusion Matrix:
[[30  0]
 [ 1 49]]

RANDOM FOREST
Accuracy: 100.0 %

Classification Report:
              precision    recall  f1-score   support

     Not CKD       1.00      1.00      1.00        30
         CKD       1.00      1.00      1.00        50

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80

Confusion Matrix:
[[30  0]
 [ 0 50]]


In [13]:
# Create stratified 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluate Logistic Regression
lr_cv_scores = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

# Evaluate Random Forest
rf_cv_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("LOGISTIC REGRESSION")
print("Fold accuracies:", lr_cv_scores)
print("Mean CV accuracy:", round(lr_cv_scores.mean() * 100, 2), "%")

print("\nRANDOM FOREST")
print("Fold accuracies:", rf_cv_scores)
print("Mean CV accuracy:", round(rf_cv_scores.mean() * 100, 2), "%")


LOGISTIC REGRESSION
Fold accuracies: [1.     1.     0.9875 1.     1.    ]
Mean CV accuracy: 99.75 %

RANDOM FOREST
Fold accuracies: [0.9875 1.     0.9875 0.975  1.    ]
Mean CV accuracy: 99.0 %


In [14]:
# Create the models directory if it does not exist
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Save the complete pipeline
model_path = models_dir / "kidney_disease_model.pkl"
joblib.dump(logistic_model, model_path)

print("Model saved successfully!")
print("Saved location:", model_path.resolve())


Model saved successfully!
Saved location: E:\Multi_Disease_Prediction_System\models\kidney_disease_model.pkl


In [15]:
# Load the saved model
kidney_model = joblib.load("../models/kidney_disease_model.pkl")

# Test on one held-out record
sample_patient = X_test.iloc[[0]]

prediction = kidney_model.predict(sample_patient)[0]
probabilities = kidney_model.predict_proba(sample_patient)[0]

print("Actual class:", "CKD" if y_test.iloc[0] == 1 else "Not CKD")
print("Predicted class:", "CKD" if prediction == 1 else "Not CKD")
print("Predicted class probabilities:")
print("Not CKD:", round(probabilities[0] * 100, 2), "%")
print("CKD:", round(probabilities[1] * 100, 2), "%")


Actual class: Not CKD
Predicted class: Not CKD
Predicted class probabilities:
Not CKD: 99.67 %
CKD: 0.33 %


In [16]:
# Check duplicate records
print("Exact duplicate rows:", df.duplicated().sum())

# Check duplicate patient identifiers
print("Duplicate IDs:", df["id"].duplicated().sum())

# Check for missing target labels
print("Missing target labels:", df["classification"].isna().sum())

# Confirm the target distribution
print("\nTarget distribution:")
print(df["classification"].value_counts())


Exact duplicate rows: 0
Duplicate IDs: 0
Missing target labels: 0

Target distribution:
classification
ckd       250
notckd    150
Name: count, dtype: Int64


In [17]:
# v1 


In [18]:
import joblib
from pathlib import Path

model_path = Path("../models/kidney_disease_model.pkl")
saved_model = joblib.load(model_path)

print("Saved model type:", type(saved_model).__name__)

if hasattr(saved_model, "feature_names_in_"):
    print("\nExpected feature order:")
    print(list(saved_model.feature_names_in_))
elif hasattr(saved_model, "named_steps"):
    print("\nPipeline steps:")
    print(saved_model.named_steps)
    print("\nFeature names may need to be checked from the training data.")
else:
    print("\nFeature names are not stored directly on the model.")

print("\nModel loaded successfully.")


Saved model type: Pipeline

Expected feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']

Model loaded successfully.


In [19]:
# v2 

In [20]:
import joblib
import numpy as np
from pathlib import Path
from collections import Counter
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

saved_model = joblib.load(Path("../models/kidney_disease_model.pkl"))

# Verify that test data contains the expected features
expected_features = list(saved_model.feature_names_in_)

print("Test data shape:", X_test.shape)
print("Expected feature count:", len(expected_features))
print("Feature sets match:", set(X_test.columns) == set(expected_features))

if set(X_test.columns) != set(expected_features):
    raise ValueError("X_test columns do not match the saved model features.")

# Align columns with the saved model's expected order
X_test_aligned = X_test[expected_features]

# Evaluate the saved model
predictions = saved_model.predict(X_test_aligned)

print("\nPrediction counts:")
print(Counter(predictions))

print("\nActual test-label counts:")
print(Counter(np.asarray(y_test).ravel()))

print("\nSaved model accuracy:")
print(f"{accuracy_score(y_test, predictions) * 100:.2f}%")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))


Test data shape: (80, 24)
Expected feature count: 24
Feature sets match: True

Prediction counts:
Counter({np.int64(1): 49, np.int64(0): 31})

Actual test-label counts:
Counter({np.int64(1): 50, np.int64(0): 30})

Saved model accuracy:
98.75%

Confusion matrix:
[[30  0]
 [ 1 49]]

Classification report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        30
           1       1.00      0.98      0.99        50

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80



In [21]:
# v3 

In [22]:
import pandas as pd

expected_features = list(saved_model.feature_names_in_)
X_test_aligned = X_test[expected_features]

predictions = saved_model.predict(X_test_aligned)

results = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "predicted": predictions
}, index=X_test_aligned.index)

# One correctly predicted sample from each class
for label in [0, 1]:
    sample_index = results[
        (results["actual"] == label) &
        (results["predicted"] == label)
    ].index[0]

    print(f"\n{'=' * 15} SAMPLE: CLASS {label} {'=' * 15}")
    print(X_test_aligned.loc[sample_index].to_dict())
    print("Actual label:", results.loc[sample_index, "actual"])
    print("Saved model prediction:", results.loc[sample_index, "predicted"])


=============== SAMPLE: CLASS 0 ===============
{'age': 44.0, 'bp': 60.0, 'sg': 1.02, 'al': 0.0, 'su': 0.0, 'rbc': 'normal', 'pc': 'normal', 'pcc': 'notpresent', 'ba': 'notpresent', 'bgr': 95.0, 'bu': 46.0, 'sc': 0.5, 'sod': 138.0, 'pot': 4.2, 'hemo': 15.0, 'pcv': 50.0, 'wc': 7700.0, 'rc': 6.3, 'htn': 'no', 'dm': 'no', 'cad': 'no', 'appet': 'good', 'pe': 'no', 'ane': 'no'}
Actual label: 0
Saved model prediction: 0

=============== SAMPLE: CLASS 1 ===============
{'age': 65.0, 'bp': 90.0, 'sg': 1.01, 'al': 4.0, 'su': 2.0, 'rbc': 'normal', 'pc': 'normal', 'pcc': 'notpresent', 'ba': 'notpresent', 'bgr': 172.0, 'bu': 82.0, 'sc': 13.5, 'sod': 145.0, 'pot': 6.3, 'hemo': 8.8, 'pcv': 31.0, 'wc': nan, 'rc': nan, 'htn': 'yes', 'dm': 'yes', 'cad': 'no', 'appet': 'good', 'pe': 'yes', 'ane': 'yes'}
Actual label: 1
Saved model prediction: 1
